<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Algoritmos Iterativos em Python — Convergência e Critérios de Parada.</div>
  <div style="font-size: 18px;"><strong>Prof.  Dr.  E. R. Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2027</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

__Apresentação__

Na Engenharia, muitas equações não podem ser resolvidas por uma única operação algébrica. Nesses casos, partimos de uma estimativa inicial, calculamos uma nova aproximação e repetimos o procedimento até que o resultado atenda a um critério de precisão. Esse procedimento recebe o nome de **algoritmo iterativo**.

**Objetivos de aprendizagem:** compreender o significado de iteração e convergência; estabelecer critérios de parada; implementar algoritmos com `while` e funções; identificar falhas de convergência; e interpretar resultados em situações de Engenharia Química e Ciências dos Materiais.

**Pré-requisitos:** variáveis, funções, estruturas condicionais, `for`, `while`, listas e dicionários. Usaremos inicialmente apenas recursos básicos do Python, sem depender de bibliotecas científicas.

__Sumário__

1. [Por que iterar?](#secao-1)
2. [Aproximações sucessivas: primeiro cálculo manual](#secao-2)
3. [Erros, tolerâncias e critérios de parada](#secao-3)
4. [Construção de um algoritmo iterativo confiável](#secao-4)
5. [Instrumentação: inversão de uma calibração não linear](#secao-5)
6. [Controle de Processos: regime permanente de um tanque aquecido](#secao-6)
7. [Processos Químicos: concentração de equilíbrio](#secao-7)
8. [Modelagem: método da bisseção](#secao-8)
9. [Ciências dos Materiais: temperatura e propriedade dependente](#secao-9)
10. [Quando o algoritmo falha?](#secao-10)
11. [Boas práticas e interpretação](#secao-11)
12. [Exercícios de consolidação](#secao-12)
13. [Síntese do notebook](#secao-13)

<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>1. Por que iterar?</strong></div>

Considere um sistema de aquecimento cujo coeficiente de perda térmica depende da própria temperatura. Não sabemos a temperatura final antes de calcular as perdas; mas, para calcular as perdas, precisamos estimar a temperatura. Uma solução consiste em **supor uma temperatura, atualizar o cálculo e repetir**.

Em termos gerais, um algoritmo de ponto fixo usa

$$x_{k+1}=g(x_k),$$

em que $x_k$ é a estimativa atual e $x_{k+1}$ é a nova estimativa. O índice $k$ conta as iterações. **Nem toda função $g$ produz convergência**: isso depende da equação, da forma de rearranjá-la e da estimativa inicial.

<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>2. Aproximações sucessivas: primeiro cálculo manual</strong></div>

__Exemplo introdutório: encontrar a raiz quadrada de 10__

Desejamos determinar o número positivo $x$ que satisfaz $x^2=10$. Uma forma de produzir aproximações é

$$x_{k+1}=\frac{1}{2}\left(x_k+\frac{10}{x_k}\right).$$

Escolhendo $x_0=3$, calculamos manualmente:

$$x_1=\frac{1}{2}\left(3+\frac{10}{3}\right)\approx 3{,}166667,$$

$$x_2=\frac{1}{2}\left(3{,}166667+\frac{10}{3{,}166667}\right)\approx 3{,}162281.$$

As aproximações se aproximam de $\sqrt{10}\approx 3{,}162278$. Agora vamos automatizar exatamente o procedimento manual.

In [9]:
valor = 10.0
x = 3.0

for k in range(1, 6):
    novo = 0.5 * (x + valor / x)
    print(f"Iteração {k}: x = {novo:.8f}")
    x = novo

Iteração 1: x = 3.16666667
Iteração 2: x = 3.16228070
Iteração 3: x = 3.16227766
Iteração 4: x = 3.16227766
Iteração 5: x = 3.16227766


**Interpretação:** cada linha representa uma aproximação calculada a partir da anterior. Repetir um número fixo de vezes ajuda a visualizar o método, mas não garante que a precisão desejada tenha sido atingida.

<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>3. Erros, tolerâncias e critérios de parada</strong></div>

A **variação absoluta entre iterações** é

$$\Delta_k=|x_{k+1}-x_k|.$$

A **variação relativa** pode ser definida, para $x_{k+1}\ne 0$, como

$$r_k=\frac{|x_{k+1}-x_k|}{|x_{k+1}|}.$$

O **resíduo** mede quanto a estimativa viola a equação original $f(x)=0$:

$$R_k=|f(x_k)|.$$

Essas grandezas não são equivalentes. Uma variação pequena entre duas aproximações **não garante**, por si só, que a equação original foi bem satisfeita. Além disso, resíduos têm unidades e escalas próprias; sua tolerância deve ser escolhida de acordo com o problema.

Um algoritmo prático deve prever: tolerância de variação, tolerância de resíduo quando aplicável, número máximo de iterações e verificação de resultados não finitos. Em problemas com variáveis de escalas muito diferentes, podem ser necessárias tolerâncias absolutas e relativas combinadas.

<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>4. Construção de um algoritmo iterativo confiável</strong></div>

__Implementação: solução iterativa com proteção__

Vamos resolver $x^2-10=0$, para $x>0$, interrompendo o cálculo apenas quando a variação **e** o resíduo estiverem suficientemente pequenos. A função também interrompe a execução quando atinge um limite de iterações.

In [10]:
import math

def raiz_por_iteracao(valor, estimativa=3.0, tol_x=1e-8,
                      tol_f=1e-8, max_iter=100):
    """Calcula a raiz positiva por aproximações sucessivas."""
    if valor <= 0 or estimativa <= 0:
        raise ValueError("Valor e estimativa devem ser positivos.")
    if tol_x <= 0 or tol_f <= 0 or max_iter < 1:
        raise ValueError("Tolerâncias e limite devem ser positivos.")

    x = float(estimativa)
    historico = []
    for k in range(1, max_iter + 1):
        novo = 0.5 * (x + valor / x)
        variacao = abs(novo - x)
        residuo = abs(novo**2 - valor)
        historico.append((k, novo, variacao, residuo))

        if not math.isfinite(novo):
            raise ArithmeticError("Resultado numérico não finito.")
        if variacao <= tol_x and residuo <= tol_f:
            return novo, historico, True
        x = novo

    return x, historico, False

solucao, historico, convergiu = raiz_por_iteracao(10)
print(f"Solução: {solucao:.9f}")
print(f"Iterações: {len(historico)} | Convergiu: {convergiu}")
print(f"Resíduo final: {historico[-1][3]:.3e}")

Solução: 3.162277660
Iterações: 4 | Convergiu: True
Resíduo final: 1.776e-15


**Discussão:** `max_iter` impede um laço infinito; `convergiu` informa o estado do cálculo; e o histórico permite examinar cada aproximação. Em aplicações industriais, uma resposta numérica sem indicação de convergência pode induzir a decisões incorretas.

<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>5. Instrumentação: inversão de uma calibração não linear</strong></div>

__Instrumentação e Controle de Qualidade — calibração não linear__

Considere um sensor hipotético de pressão cuja curva de calibração, obtida experimentalmente, seja

$$I(P)=4+1{,}5P+0{,}02P^2,$$

em que $I$ é a corrente em mA e $P$ é a pressão em bar, válida no intervalo de calibração de 0 a 10 bar. Para uma corrente medida de 12 mA, queremos determinar $P$.

A equação a resolver é

$$f(P)=4+1{,}5P+0{,}02P^2-12=0.$$

Rearranjando para uma iteração:

$$P_{k+1}=\frac{12-4-0{,}02P_k^2}{1{,}5}.$$

**Cálculo manual:** com $P_0=5$ bar, obtemos

$$P_1=\frac{8-0{,}02(5)^2}{1{,}5}=5\ \mathrm{bar}.$$

Neste caso particular, a estimativa inicial já satisfaz a equação. Para observar o processo iterativo, utilizaremos $P_0=4$ bar. Os coeficientes são **didáticos**, não uma especificação de transmissor comercial.

In [11]:
corrente = 12.0  # mA
pressao = 4.0  # bar: estimativa inicial
tolerancia = 1e-8

for k in range(1, 101):
    nova_pressao = (corrente - 4.0 - 0.02 * pressao**2) / 1.5
    residuo = abs(4.0 + 1.5 * nova_pressao + 0.02 * nova_pressao**2 - corrente)
    print(f"{k:2d} | P = {nova_pressao:.7f} bar | resíduo = {residuo:.2e} mA")
    if abs(nova_pressao - pressao) < tolerancia and residuo < tolerancia:
        pressao = nova_pressao
        break
    pressao = nova_pressao
else:
    print("Atenção: limite de iterações atingido.")

print(f"Pressão estimada: {pressao:.6f} bar")

 1 | P = 5.1200000 bar | resíduo = 2.04e-01 mA
 2 | P = 4.9838080 bar | resíduo = 2.75e-02 mA
 3 | P = 5.0021554 bar | resíduo = 3.66e-03 mA
 4 | P = 4.9997125 bar | resíduo = 4.89e-04 mA
 5 | P = 5.0000383 bar | resíduo = 6.52e-05 mA
 6 | P = 4.9999949 bar | resíduo = 8.69e-06 mA
 7 | P = 5.0000007 bar | resíduo = 1.16e-06 mA
 8 | P = 4.9999999 bar | resíduo = 1.54e-07 mA
 9 | P = 5.0000000 bar | resíduo = 2.06e-08 mA
10 | P = 5.0000000 bar | resíduo = 2.75e-09 mA
11 | P = 5.0000000 bar | resíduo = 3.66e-10 mA
Pressão estimada: 5.000000 bar


**Interpretação:** a leitura elétrica é convertida em uma grandeza de processo por inversão da curva de calibração. Antes de usar o resultado, o engenheiro deve verificar se a corrente e a pressão calculada estão dentro da faixa em que a calibração é válida.

<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>6. Controle de Processos: regime permanente de um tanque aquecido</strong></div>

__Controle de Processos — temperatura de equilíbrio de um tanque__

Considere um tanque perfeitamente misturado recebendo uma potência térmica constante $Q=1000$ W e perdendo calor para o ambiente a $T_a=25$ °C. Admitamos, apenas para estudo, que o coeficiente global de perdas dependa da temperatura:

$$UA(T)=20+0{,}1(T-25)\quad \mathrm{W/K}.$$

No regime permanente, o balanço de energia é

$$Q=UA(T)(T-T_a).$$

A temperatura desconhecida aparece dos dois lados. Podemos escrever

$$T_{k+1}=25+\frac{1000}{20+0{,}1(T_k-25)}.$$

**Primeira aproximação manual:** escolhendo $T_0=60$ °C,

$$UA(T_0)=20+0{,}1(60-25)=23{,}5\ \mathrm{W/K},$$

$$T_1=25+\frac{1000}{23{,}5}\approx67{,}55\ ^\circ\mathrm{C}.$$

O modelo representa uma condição de equilíbrio simplificada, não a resposta dinâmica nem um projeto de controle.

In [12]:
potencia = 1000.0  # W
t_ambiente = 25.0  # °C
temperatura = 60.0  # °C
convergiu = False

for k in range(1, 101):
    ua = 20.0 + 0.1 * (temperatura - t_ambiente)
    nova = t_ambiente + potencia / ua
    ua_novo = 20.0 + 0.1 * (nova - t_ambiente)
    residuo = abs(potencia - ua_novo * (nova - t_ambiente))
    if abs(nova - temperatura) < 1e-8 and residuo < 1e-6:
        temperatura = nova
        convergiu = True
        break
    temperatura = nova

print(f"Temperatura de equilíbrio: {temperatura:.4f} °C")
print(f"Resíduo energético: {residuo:.3e} W")
print(f"Iterações: {k} | Convergiu: {convergiu}")

Temperatura de equilíbrio: 66.4214 °C
Resíduo energético: 2.075e-08 W
Iterações: 13 | Convergiu: True


**Interpretação:** o resultado fornece um ponto de operação estimado. Em Controle de Processos Aplicado, esse ponto pode servir como referência para estudar perturbações, linearização e resposta dinâmica em notebooks posteriores.

<div id="secao-7" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>7. Processos Químicos: concentração de equilíbrio</strong></div>

__Processos Químicos de Fabricação — concentração em equilíbrio__

Em um exemplo didático de processo com consumo e reposição de um componente, suponha que a concentração de equilíbrio $C$ (mol/L) satisfaça

$$C+0{,}2C^2=2.$$

A equação pode representar, de forma simplificada, um balanço algébrico com um termo não linear. Não corresponde a uma cinética industrial específica. Para resolver por iteração, rearranjamos:

$$C_{k+1}=\frac{2}{1+0{,}2C_k}.$$

Com $C_0=1$ mol/L, o cálculo manual fornece

$$C_1=\frac{2}{1+0{,}2(1)}\approx1{,}6667\ \mathrm{mol/L}.$$

Observe que a próxima estimativa não é necessariamente melhor em todos os problemas: é preciso verificar a convergência.

In [13]:
concentracao = 1.0  # mol/L
for k in range(1, 101):
    nova = 2.0 / (1.0 + 0.2 * concentracao)
    residuo = abs(nova + 0.2 * nova**2 - 2.0)
    if abs(nova - concentracao) < 1e-9 and residuo < 1e-9:
        concentracao = nova
        break
    concentracao = nova
else:
    print("Não houve convergência no limite estabelecido.")

print(f"Concentração: {concentracao:.7f} mol/L")
print(f"Resíduo do balanço: {residuo:.2e}")

Concentração: 1.5311289 mol/L
Resíduo do balanço: 7.63e-11


<div id="secao-8" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>8. Modelagem: método da bisseção</strong></div>

__Método da bisseção — solução robusta com intervalo__

Nem sempre é possível encontrar um rearranjo de ponto fixo que convirja. O **método da bisseção** parte de uma função contínua $f(x)$ e de um intervalo $[a,b]$ em que $f(a)$ e $f(b)$ tenham sinais opostos. Calculamos o ponto médio

$$m=\frac{a+b}{2}$$

e conservamos a metade do intervalo que mantém a mudança de sinal. Para a equação $f(x)=x^2-10$, temos $f(3)=-1$ e $f(4)=6$. A primeira aproximação é $m_1=3{,}5$ e $f(3{,}5)=2{,}25$, logo o novo intervalo é $[3;3{,}5]$.

A bisseção é mais lenta que alguns outros métodos, mas possui garantia de aproximação a uma raiz sob essas hipóteses. Um intervalo final pequeno limita a incerteza de localização da raiz.

In [14]:
def bissecao(funcao, a, b, tol_x=1e-8, tol_f=1e-8, max_iter=100):
    fa, fb = funcao(a), funcao(b)
    if not (math.isfinite(fa) and math.isfinite(fb)):
        raise ValueError("Valores da função devem ser finitos.")
    if fa == 0:
        return a, 0, True
    if fb == 0:
        return b, 0, True
    if fa * fb > 0:
        raise ValueError("O intervalo não apresenta mudança de sinal.")

    for k in range(1, max_iter + 1):
        meio = (a + b) / 2
        fm = funcao(meio)
        if not math.isfinite(fm):
            raise ArithmeticError("Valor não finito durante a iteração.")
        if abs(fm) <= tol_f or (b - a) / 2 <= tol_x:
            return meio, k, True
        if fa * fm < 0:
            b, fb = meio, fm
        else:
            a, fa = meio, fm
    return (a + b) / 2, max_iter, False

raiz, iteracoes, ok = bissecao(lambda x: x**2 - 10, 3, 4)
print(f"Raiz: {raiz:.9f} | Iterações: {iteracoes} | Convergiu: {ok}")
print(f"Resíduo: {abs(raiz**2 - 10):.3e}")

Raiz: 3.162277661 | Iterações: 27 | Convergiu: True
Resíduo: 6.929e-09


**Atenção:** no código, o retorno `True` significa que **ao menos um** dos dois critérios de parada foi satisfeito (resíduo ou largura do intervalo). Para aplicações críticas, convém registrar qual deles determinou a parada e definir tolerâncias consistentes com as unidades e a precisão física do problema.

<div id="secao-9" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>9. Ciências dos Materiais: temperatura e propriedade dependente</strong></div>

__Ciências dos Materiais — estimativa de temperatura por propriedade medida__

Considere uma propriedade elétrica fictícia $R(T)$, medida em ohms, que dependa da temperatura conforme

$$R(T)=100[1+0{,}004(T-20)+0{,}00001(T-20)^2].$$

Um experimento fornece $R=130$ ohms. Qual temperatura corresponde a essa leitura? O problema é uma **inversão de curva de caracterização**, semelhante à calibração de sensores ou à interpretação de propriedades de materiais. Para ilustrar um método diferente do ponto fixo, aplicaremos a bisseção ao intervalo de 20 a 100 °C.

In [15]:
def resistencia(temperatura):
    delta = temperatura - 20.0
    return 100.0 * (1.0 + 0.004 * delta + 0.00001 * delta**2)

medicao = 130.0  # ohms
funcao = lambda temperatura: resistencia(temperatura) - medicao

temperatura, n_iter, ok = bissecao(funcao, 20.0, 100.0,
                                  tol_x=1e-7, tol_f=1e-7)
print(f"Temperatura estimada: {temperatura:.4f} °C")
print(f"Resistência recalculada: {resistencia(temperatura):.6f} ohms")
print(f"Iterações: {n_iter} | Convergiu: {ok}")

Temperatura estimada: 84.5751 °C
Resistência recalculada: 130.000000 ohms
Iterações: 25 | Convergiu: True


**Interpretação:** a estimativa só é defensável dentro do intervalo de caracterização e enquanto a relação entre propriedade e temperatura permanecer válida. Materiais reais podem apresentar histerese, mudanças de fase e outros efeitos que exigem modelos mais completos.

<div id="secao-10" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>10. Quando o algoritmo falha?</strong></div>

### Um exemplo de divergência

Para resolver $x^2-2=0$, alguém poderia propor a iteração

$$x_{k+1}=x_k^2-2+x_k.$$

Embora qualquer raiz da equação original seja um ponto fixo, essa forma iterativa pode divergir. Partindo de $x_0=2$, as aproximações crescem rapidamente. Vamos limitar a execução e identificar o problema.

In [16]:
x = 2.0
for k in range(1, 8):
    x = x**2 - 2 + x
    print(f"Iteração {k}: x = {x:.3e}")
    if not math.isfinite(x) or abs(x) > 1e12:
        print("Processo interrompido: crescimento excessivo.")
        break

Iteração 1: x = 4.000e+00
Iteração 2: x = 1.800e+01
Iteração 3: x = 3.400e+02
Iteração 4: x = 1.159e+05
Iteração 5: x = 1.344e+10
Iteração 6: x = 1.807e+20
Processo interrompido: crescimento excessivo.


**Discussão:** o problema não está no comando `for` ou `while`, mas na escolha do algoritmo. Uma equação correta não garante que qualquer fórmula de atualização funcione. A estimativa inicial e o comportamento local de $g(x)$ também importam.

<div id="secao-11" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>11. Boas práticas e interpretação</strong></div>

Ao analisar uma solução iterativa, o engenheiro deve verificar se a equação representa corretamente o fenômeno, se os dados e unidades são consistentes, se o método é adequado, se o resultado permanece no domínio físico e se os critérios de parada são atendidos. É recomendável conservar o histórico de iterações e informar a tolerância, o resíduo e o número de passos.

**Diferença importante:** erro verdadeiro exige conhecer a solução exata ou uma referência confiável; variação entre iterações e resíduo são indicadores computáveis, mas não substituem automaticamente esse erro. Em problemas de controle, um cálculo que não converge deve ser sinalizado, nunca tratado silenciosamente como resultado válido.

<div id="secao-12" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>12. Exercícios de consolidação</strong></div>

**Exercício 1 — Instrumentação:** altere a corrente do exemplo de calibração para 10 mA. Determine a pressão e verifique o resíduo. Discuta se o resultado pertence à faixa de calibração.

**Exercício 2 — Controle de Processos:** no tanque aquecido, modifique a potência para 800 W. Calcule a nova temperatura de equilíbrio e compare com a condição anterior.

**Exercício 3 — Modelagem:** utilize a bisseção para resolver $x^3-x-2=0$ no intervalo $[1,2]$. Compare o resultado obtido com duas tolerâncias diferentes e registre o número de iterações.

**Exercício 4 — Controle de Qualidade:** uma curva didática de calibração é $S(C)=2+0{,}8C+0{,}05C^2$. Para uma leitura $S=8$, determine a concentração positiva $C$ com bisseção no intervalo de 0 a 10. Apresente unidades apenas após definir a natureza da medição.

**Exercício 5 — Ciências dos Materiais:** repita o exemplo da resistência para uma leitura de 120 ohms. Explique como a faixa de temperatura escolhida influencia a validade da resposta.

**Desafio integrador:** escreva uma função que receba a potência térmica e a temperatura inicial do tanque, devolva a temperatura de equilíbrio, o número de iterações, o resíduo energético e um indicador de convergência. Experimente três condições operacionais e apresente os resultados em uma tabela simples.

<div id="secao-13" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:22px;"><strong>13. Síntese do notebook</strong></div>

Neste notebook, partimos de aproximações manuais e construímos algoritmos capazes de decidir quando interromper os cálculos. Estudamos ponto fixo, bisseção, variação entre iterações, resíduo, tolerância e proteção contra falhas. Aplicamos esses conceitos à calibração de instrumentos, ao equilíbrio térmico, a balanços não lineares e à caracterização de materiais.

O passo seguinte da coleção será trabalhar com **leitura e escrita de arquivos**, para registrar medições, históricos de iteração e resultados de simulações.

<div style="text-align: center; font-size: 12px; color: gray;
            margin-top: 40px; padding-top: 15px;
            border-top: 1px solid #dddddd;">

  <div style="text-align: center; white-space: nowrap;">
    Material desenvolvido pelo Grupo de Pesquisas em Modelagem Computacional da UESC.
  </div>

  <div style="text-align: center; white-space: nowrap; margin-top: 15px;">
    © 2026 — Todos os direitos reservados. Uso comercial somente mediante autorização.
  </div>

</div>